---
title: Convergence Angle Notebook
authors: [gvarnavides]
date: 2026-05-28
---

In [ ]:
%matplotlib widget

import abtem
import numpy as np
import quantem as em
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

from matplotlib.gridspec import GridSpec
import ipywidgets

plt.rcParams['text.color']='white'
plt.rcParams['xtick.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.labelcolor'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'white'
# plt.rcParams.update({
#     "text.usetex": True,
#     "text.latex.preamble": r"\usepackage{amsmath}"
# })

abtem.config.set({"dask.lazy":False});


In [2]:
def array_to_scaled_rgba(array,vmin=0.02,vmax=0.98):
    if np.iscomplexobj(array):
        scaled_amplitude = np.abs(array)
        scaled_angle = np.angle(array)
    else:
        scaled_amplitude = array
        scaled_angle = None

    if scaled_amplitude.std() > 1e-12:
        vmin, vmax = np.quantile(scaled_amplitude,(vmin,vmax))

        scaled_amplitude = (scaled_amplitude.clip(vmin,vmax) - vmin) / (vmax-vmin)
    else:
        scaled_amplitude = np.ones_like(scaled_amplitude)

    rgba = em.visualization.visualization_utils.array_to_rgba(
        scaled_amplitude,
        scaled_angle
    )

    return rgba

def return_probe_arrays(semiangle_cutoff,aberrations=None,**kwargs):
    """ """
    ctf = abtem.CTF(
        semiangle_cutoff=semiangle_cutoff,
        sampling=(0.125,0.125),
        gpts=(128,128),
        energy=300e3,
        aberration_coefficients=aberrations,
        **kwargs,
    )
    
    fourier_probe = ctf.to_diffraction_patterns(
        gpts=ctf.gpts
    ).array
    real_probe = ctf.to_point_spread_functions(
        gpts=ctf.gpts,
        extent=ctf.extent
    ).array
    
    return ctf, fourier_probe, real_probe    

In [3]:
width = 620
aspect_ratio = 0.45
height = int(width * aspect_ratio)
dpi = 72
with plt.ioff():
    fig,axs = plt.subplots(1,2,figsize=(width/dpi,height/dpi),dpi=dpi)

ctf, fourier_probe, real_probe  = return_probe_arrays(
    semiangle_cutoff=20,
)

rgb_fourier_probe = array_to_scaled_rgba(fourier_probe,vmin=0.001,vmax=0.999)
im_fourier = axs[0].imshow(rgb_fourier_probe)

rgb_real_probe = array_to_scaled_rgba(real_probe,vmin=0.001,vmax=0.999)
im_real = axs[1].imshow(rgb_real_probe)

scalebar_real = em.visualization.ScalebarConfig(ctf.sampling[0],units=r'$\AA$')
scalebar_fourier = em.visualization.ScalebarConfig(ctf.angular_sampling[0],units='mrad')

bars = [scalebar_fourier, scalebar_real]
titles= ["Fourier-space probe", "real-space probe"]

for ax, bar, title in zip(axs,bars, titles):
    ax.patch.set_alpha(0)
    ax.set(xticks=[],yticks=[],title=title)
    divider = make_axes_locatable(ax)
    ax_cb = divider.append_axes("right", size="5%", pad="2.5%")
    em.visualization.visualization_utils.add_arg_cbar_to_ax(fig,ax_cb)
    em.visualization.visualization_utils.add_scalebar_to_ax(
        ax,
        rgb_fourier_probe.shape[1],
        bar.sampling,
        bar.length,
        bar.units,
        bar.width_px,
        bar.pad_px,
        bar.color,
        bar.loc,
    )

fig.tight_layout()
fig.canvas.resizable = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.layout.width = f'{width}px'
fig.canvas.toolbar_position = 'bottom'
fig.patch.set_alpha(0)
None

In [4]:
def update_aperture(change):
    """ """
    semiangle_cutoff = change['new']
    ctf, fourier_probe, real_probe  = return_probe_arrays(
        semiangle_cutoff=semiangle_cutoff,
    )


    rgb_fourier_probe = array_to_scaled_rgba(fourier_probe,vmin=0.001,vmax=0.999)
    im_fourier.set_data(rgb_fourier_probe)
    
    rgb_real_probe = array_to_scaled_rgba(real_probe,vmin=0.001,vmax=0.999)
    im_real.set_data(rgb_real_probe)
    fig.canvas.draw_idle()
    return None

style = {
    'description_width': 'initial',
}

layout = ipywidgets.Layout(width='450px',height='30px')

semiangle_slider = ipywidgets.FloatSlider(
    min=5,
    max=50,
    step=0.5,
    value=20,
    layout=layout,
    style=style,
    description="convergence semi-angle [mrad]",
) 
semiangle_slider.observe(update_aperture, 'value')

In [5]:
#| label: app:convergence_angle_widget
display(
    ipywidgets.VBox(
        [
            semiangle_slider,
            fig.canvas
        ],
        layout=ipywidgets.Layout(
            align_items="center"
        )
    )
)